In [12]:
import pandas as pd
training_data_file_path=r"C:\Users\khmam\Documents\GitHub\Predective-Industrial-Maintainance\data_sets\ai4i2020.csv"
train_data=pd.read_csv(training_data_file_path)
features=['Type','Air temperature','Process temperature','Rotational speed','Torque','Tool wear']
X=train_data[features]
y_features=['Machine failure','TWF','HDF','PWF','OSF','RNF']
y=train_data[y_features]
X.head()

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
0,M,298.1,308.6,1551,42.8,0
1,L,298.2,308.7,1408,46.3,3
2,L,298.1,308.5,1498,49.4,5
3,L,298.2,308.6,1433,39.5,7
4,L,298.2,308.7,1408,40.0,9


In [3]:
#Checking for empty values
X.isnull().sum()

Type                   0
Air temperature        0
Process temperature    0
Rotational speed       0
Torque                 0
Tool wear              0
dtype: int64

In [4]:
#Splitting data into training and validation sets
from sklearn.model_selection import train_test_split
X_train,X_valid,y_train,y_valid=train_test_split(X,y,train_size=0.8,test_size=0.2)

In [ ]:
#Preprocessing numerical data
from sklearn.preprocessing import StandardScaler,MinMaxScaler

scaler=MinMaxScaler()
X_train_scaled=X_train.copy()
X_valid_scaled=X_valid.copy()
numerical_cols=['Air temperature','Process temperature','Rotational speed','Torque','Tool wear']

X_train_scaled[numerical_cols]=scaler.fit_transform(X_train[numerical_cols])
X_valid_scaled[numerical_cols]=scaler.transform(X_valid[numerical_cols])


In [9]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform
oh_train_cols = encoder.fit_transform(X_train_scaled[['Type']])
oh_train_cols_df = pd.DataFrame(oh_train_cols, columns=encoder.get_feature_names_out(['Type']))
oh_train_cols_df.index = X_train_scaled.index  # Preserve the index

# Transform validation data
oh_valid_cols = encoder.transform(X_valid_scaled[['Type']])
oh_valid_cols_df = pd.DataFrame(oh_valid_cols, columns=encoder.get_feature_names_out(['Type']))
oh_valid_cols_df.index = X_valid_scaled.index  # Preserve the index

# Dropping 'Type' feature and concatenating with encoded columns
encoded_X_train = X_train_scaled.drop('Type', axis=1)
encoded_X_train = pd.concat([encoded_X_train, oh_train_cols_df], axis=1)

encoded_X_valid = X_valid_scaled.drop('Type', axis=1)
encoded_X_valid = pd.concat([encoded_X_valid, oh_valid_cols_df], axis=1)
encoded_X_train.head()

,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Type_H,Type_L,Type_M
5581,0.782609,0.790123,0.166473,0.601648,0.146245,0.0,0.0,1.0
6461,0.576087,0.530864,0.196158,0.440934,0.375494,0.0,1.0,0.0
9532,0.423913,0.592593,0.125728,0.616758,0.367589,0.0,1.0,0.0
5991,0.586957,0.617284,0.144354,0.502747,0.845850,0.0,1.0,0.0
525,0.228261,0.456790,0.173458,0.572802,0.209486,0.0,1.0,0.0


In [7]:
#Model training and prediction
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
model=XGBClassifier(learning_rate=0.008,n_estimators=500,eval_metric='logloss')
model.fit(encoded_X_train,y_train)
prediction=model.predict(encoded_X_valid)
print(accuracy_score(prediction,y_valid))

0.9805


In [8]:
import joblib
joblib.dump(model,"xgboost_model.joblib")
joblib.dump(scaler,"standardScaler.joblib")
joblib.dump(encoder,"oneHotEncoder.joblib")



['oneHotEncoder.joblib']